In [1]:
import pandas as pd

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix


In [ ]:

def find_duplicate_column(df):
    duplicate=[]
    checked=set()
    cols=df.columns.tolist()
    for i,col1 in enumerate(cols):
        if col1 in checked :
            continue
        group=[col1]
        for col2 in cols[i+1:]:
            if col2 in checked :
                continue
            if df[col1].equals(df[col2]):
                group.append(col2)
                checked.add(col2)
            if len(group) >1:
                duplicate.append(group)
                checked.add(col1)
    return duplicate

train=pd.read_csv("../data/processed/train.csv")
val=pd.read_csv("../data/processed/validation.csv")
test=pd.read_csv("../data/processed/test.csv")

for d in [train,test,val]:
    d['Label']=d['Label'].str.strip().str.lower()
    unnamed_cols=[c for c in d.columns if c.startswith('Unnamed:')]
    d.drop(columns=unnamed_cols,inplace=True,errors='ignore')

dup_groups=find_duplicate_column(train)
cols_to_drop=[col for group in dup_groups for col in group[1:]]

train=train.drop(columns=cols_to_drop)
test=test.drop(columns=cols_to_drop)
val=val.drop(columns=cols_to_drop)






In [ ]:
print(train["Label"].unique())

In [2]:

train=pd.read_csv("../data/processed/train.csv")
test=pd.read_csv("../data/processed/test.csv")

In [3]:
label_col='label'

X_train=train.drop(columns=[label_col])
y_train =train[label_col]

X_test=test.drop(columns=[label_col])
y_test=test[label_col]




In [ ]:
print(X_train.shape,y_train.shape)

In [7]:
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
y_train=le.fit_transform(train["label"])
y_test=le.transform(test["label"])

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_val_scaled=scaler.transform(X_val)
X_test_scaled=scaler.transform(X_test)

In [ ]:

clf=RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1,class_weight='balanced')
clf.fit(X_train,y_train)

In [ ]:
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred,target_names=le.classes_))

In [ ]:
pca=PCA(n_components=15,random_state=42)
X_train_pca=pca.fit_transform(X_train_scaled)
X_val_pca=pca.transform(X_val_scaled)
X_test_pca=pca.transform(X_test_scaled)

In [ ]:
clf_pca=RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1,class_weight='balanced')
clf_pca.fit(X_train_pca,y_train)

y_test_pred_pca=clf_pca.predict(X_test_pca)
print(classification_report(y_test,y_test_pred_pca,target_names=le.classes_))